<a href="https://colab.research.google.com/github/Charan621k/Real-Time-Number-Plate-Detection-Using-Deep-Learning-and-OCR-/blob/main/manipulator_kinematics_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Manipulator Kinematics
## Lab Notebook — OpenManipulator-X with Robotics Toolbox

**Advanced Control Engineering · MSc Robotics**

> **Note:** This notebook requires `roboticstoolbox-python` and `spatialmath-python`.
> Both are installed in the course Docker container.
> Run all cells in order.

---

### Learning objectives

1. Build a robot model from DH parameters using Robotics Toolbox.
2. Compute **forward kinematics** (FK): joint angles → end-effector pose.
3. Solve **inverse kinematics** (IK): end-effector pose → joint angles.
4. Compute the **Jacobian** and manipulability.
5. Identify **singular configurations**.

In [ ]:
pip install roboticstoolbox-python

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import roboticstoolbox as rtb
from spatialmath import SE3

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
print('Imports OK. rtb version:', rtb.__version__)

---
## Part 1 — Build the robot model

### 1.1 DH parameters

The OpenManipulator-X has 4 revolute joints.
Its DH table (from the manufacturer):

| Joint | $\theta_i$ | $d_i$ [m] | $a_i$ [m] | $\alpha_i$ | offset |
|---|---|---|---|---|---|
| 1 | $q_1$ | 0.077 | 0 | $+\pi/2$ | 0 |
| 2 | $q_2$ | 0 | 0.130 | 0 | $-\pi/2$ |
| 3 | $q_3$ | 0 | 0.124 | 0 | $+\pi/2$ |
| 4 | $q_4$ | 0 | 0.126 | 0 | 0 |

The `offset` shifts the zero of $q_i$
so that $q = (0, 0, 0, 0)$ corresponds to the arm pointing straight up.

In [ ]:
robot = rtb.DHRobot(
    [
        rtb.RevoluteDH(d=0.077, a=0,     alpha= np.pi/2, offset=0),
        rtb.RevoluteDH(d=0,     a=0.130, alpha=0,        offset=-np.pi/2),
        rtb.RevoluteDH(d=0,     a=0.124, alpha=0,        offset= np.pi/2),
        rtb.RevoluteDH(d=0,     a=0.126, alpha=0,        offset=0),
    ],
    name='OpenManipulator-X'
)

print(robot)

### 1.2 Joint limits

Set the physical joint limits of the Dynamixel XM430 actuators.

In [ ]:
# Joint limits [rad]
qlim = np.deg2rad([
    [-180, 180],   # joint 1
    [ -90,  70],   # joint 2
    [ -70,  70],   # joint 3
    [-135, 135],   # joint 4
])

for i, link in enumerate(robot.links):
    link.qlim = qlim[i]

print('Joint limits set (degrees):')
for i, link in enumerate(robot.links):
    print(f'  q{i+1}: [{np.rad2deg(link.qlim[0]):.0f}, {np.rad2deg(link.qlim[1]):.0f}] deg')

---
## Part 2 — Forward kinematics

### 2.1 FK at home configuration

`robot.fkine(q)` returns an `SE3` object:
a $4\times4$ homogeneous transformation matrix.

In [ ]:
q_home = np.zeros(4)   # arm pointing straight up

T_home = robot.fkine(q_home)
print('FK at home configuration:')
print(T_home)
print()
print('End-effector position (x, y, z) [m]:', T_home.t)
print('Expected z ≈ 0.077 + 0.130 + 0.124 + 0.126 =', 0.077+0.130+0.124+0.126, 'm')

### 2.2 FK at different configurations

Compute FK for three configurations and print the end-effector position.

In [ ]:
configs = {
    'home (all zero)':       np.zeros(4),
    'q2 = -90 deg':          np.deg2rad([0, -90, 0, 0]),
    'q1=45, q2=-45, q3=30':  np.deg2rad([45, -45, 30, 0]),
}

print(f'{"Configuration":<30}  {"x [m]":>8}  {"y [m]":>8}  {"z [m]":>8}')
print('-' * 60)
for name, q in configs.items():
    T = robot.fkine(q)
    p = T.t
    print(f'{name:<30}  {p[0]:>8.4f}  {p[1]:>8.4f}  {p[2]:>8.4f}')

### 2.3 Visualise the robot

Plot the robot arm at a given configuration.

In [ ]:
q_demo = np.deg2rad([0, -60, 45, 20])

fig = robot.plot(q_demo, backend='matplotlib', block=False)
plt.suptitle(f'OpenManipulator-X at q = {np.round(np.rad2deg(q_demo)).astype(int)} deg')
plt.tight_layout()
plt.show()

### 2.4 Task: sweep joint 2

Sweep $q_2$ from $-80°$ to $+60°$ while keeping other joints at zero.
Plot the end-effector $(x, z)$ position to visualise the workspace cross-section.

In [ ]:
q2_range = np.deg2rad(np.linspace(-80, 60, 100))

# YOUR CODE HERE
# For each q2, compute FK and store the end-effector (x, z) position.
# Hint: q = np.array([0, q2_val, 0, 0])
#        T = robot.fkine(q)
#        x, z = T.t[0], T.t[2]

xs = []  # replace with computed values
zs = []  # replace with computed values

# Plot
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(xs, zs, 'steelblue', lw=2)
ax.set_xlabel('x [m]')
ax.set_ylabel('z [m]')
ax.set_title('End-effector path as $q_2$ sweeps from $-80°$ to $+60°$')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

---
## Part 3 — Inverse kinematics

### 3.1 IK with Levenberg-Marquardt solver

`robot.ik_LM(T, q0)` uses the Levenberg-Marquardt iterative solver.
It returns a solution object with fields:
- `sol.q`: joint angles
- `sol.success`: True if converged
- `sol.iterations`: number of iterations

In [ ]:
# Target pose: end-effector at (0.25, 0.0, 0.15), no rotation
T_target = SE3(0.25, 0.0, 0.15)

q_init = np.deg2rad([0, -30, 30, 0])   # initial guess

sol = robot.ik_LM(T_target, q0=q_init)

print('IK result:')
print(f'  Success:    {sol.success}')
print(f'  Iterations: {sol.iterations}')
print(f'  q solution: {np.round(np.rad2deg(sol.q), 2)} deg')

# Verify by FK
if sol.success:
    T_check = robot.fkine(sol.q)
    err = np.linalg.norm(T_check.t - T_target.t)
    print(f'  FK position error: {err*1000:.3f} mm')

### 3.2 Multiple initial guesses

IK can have multiple solutions.
Try different initial guesses and collect all distinct solutions.

In [ ]:
T_target = SE3(0.20, 0.05, 0.18)

# YOUR CODE HERE
# Try at least 5 different initial guesses.
# Collect solutions where sol.success == True.
# Print q in degrees for each unique solution.
# Two solutions are 'distinct' if ||q1 - q2|| > 0.1 rad.

initial_guesses = [
    np.deg2rad([  0, -30,  30,  0]),
    np.deg2rad([  0, -60,  60,  0]),
    np.deg2rad([ 30, -45,  20, 10]),
    np.deg2rad([-30, -20,  40, -10]),
    np.deg2rad([  0,   0,  -30,  0]),
]

solutions = []
for q0 in initial_guesses:
    sol = robot.ik_LM(T_target, q0=q0)
    if sol.success:
        # Check if this solution is new
        is_new = all(np.linalg.norm(sol.q - s) > 0.1 for s in solutions)
        if is_new:
            solutions.append(sol.q)

print(f'Found {len(solutions)} distinct IK solution(s):')
for i, q in enumerate(solutions):
    T_check = robot.fkine(q)
    err = np.linalg.norm(T_check.t - T_target.t) * 1000
    print(f'  Solution {i+1}: {np.round(np.rad2deg(q), 1)} deg  (FK error: {err:.2f} mm)')

---
## Part 4 — The Jacobian

### 4.1 Compute the Jacobian

`robot.jacob0(q)` returns the $6\times n$ Jacobian in the base frame.
Rows 0--2: linear velocity mapping.
Rows 3--5: angular velocity mapping.

In [ ]:
q_test = np.deg2rad([0, -45, 45, 0])

J = robot.jacob0(q_test)

print('Jacobian J(q) at q =', np.round(np.rad2deg(q_test)).astype(int), 'deg:')
print(np.round(J, 4))
print()
print('Shape:', J.shape, '  (6 x n, where n=4 joints)')
print()
print('Linear velocity part J_v (rows 0-2):')
print(np.round(J[:3, :], 4))
print()
print('Angular velocity part J_w (rows 3-5):')
print(np.round(J[3:, :], 4))

### 4.2 Resolved-rate motion

Given a desired end-effector velocity $\dot x$,
compute the required joint velocities $\dot q = J^+ \dot x$.

In [ ]:
q_test = np.deg2rad([0, -45, 45, 0])
J = robot.jacob0(q_test)
J_v = J[:3, :]   # linear velocity part only (3x4)

# Desired end-effector velocity: move in +x at 0.05 m/s
xdot_d = np.array([0.05, 0.0, 0.0])   # [vx, vy, vz] m/s

# YOUR CODE HERE
# Compute the pseudo-inverse of J_v and find the joint velocities.
# Hint: np.linalg.pinv(J_v)

J_v_pinv = None   # replace
qdot = None       # replace: J_v_pinv @ xdot_d

print('Desired EE velocity:', xdot_d, 'm/s')
print('Required joint velocities [rad/s]:', np.round(qdot, 4))
print('Required joint velocities [deg/s]:', np.round(np.rad2deg(qdot), 2))

# Verify: J_v @ qdot should equal xdot_d
xdot_check = J_v @ qdot
print('Verification J_v @ qdot:', np.round(xdot_check, 6))

### 4.3 Manipulability

Manipulability $w(q) = \sqrt{\det(J J^\top)}$ measures how far from singular a configuration is.

In [ ]:
q2_range = np.deg2rad(np.linspace(-80, 60, 200))

manip = []
for q2 in q2_range:
    q = np.array([0, q2, 0, 0])
    manip.append(robot.manipulability(q))

manip = np.array(manip)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(np.rad2deg(q2_range), manip, 'steelblue', lw=2)
ax.axhline(0, color='crimson', ls='--', lw=1, label='singular ($w=0$)')
ax.fill_between(np.rad2deg(q2_range), 0, manip,
                alpha=0.15, color='steelblue')
ax.set_xlabel('$q_2$ [deg]')
ax.set_ylabel('manipulability $w(q)$')
ax.set_title('Manipulability vs $q_2$ (other joints at zero)')
ax.legend()
plt.tight_layout()
plt.show()

idx_max = np.argmax(manip)
print(f'Maximum manipulability: {manip[idx_max]:.4f} at q2 = {np.rad2deg(q2_range[idx_max]):.1f} deg')
print(f'Manipulability at home (q2=0): {robot.manipulability(np.zeros(4)):.4f}')

---
## Part 5 — Singularities

### 5.1 Finding singular configurations

A configuration is singular when $\det(J J^\top) \approx 0$.
Find configurations where manipulability drops below a threshold.

In [ ]:
# Sweep q2 and q3 jointly
N = 60
q2_vals = np.deg2rad(np.linspace(-80, 60, N))
q3_vals = np.deg2rad(np.linspace(-70, 70, N))

W = np.zeros((N, N))
for i, q2 in enumerate(q2_vals):
    for j, q3 in enumerate(q3_vals):
        q = np.array([0, q2, q3, 0])
        W[i, j] = robot.manipulability(q)

fig, ax = plt.subplots(figsize=(8, 6))
c = ax.contourf(np.rad2deg(q3_vals), np.rad2deg(q2_vals), W,
                levels=20, cmap='RdYlGn')
plt.colorbar(c, ax=ax, label='manipulability $w(q)$')
ax.contour(np.rad2deg(q3_vals), np.rad2deg(q2_vals), W,
           levels=[0.02], colors='k', linewidths=2)
ax.set_xlabel('$q_3$ [deg]')
ax.set_ylabel('$q_2$ [deg]')
ax.set_title('Manipulability map ($q_1=q_4=0$)\nBlack contour: near-singular region ($w < 0.02$)')
plt.tight_layout()
plt.show()

### 5.2 What happens at a singularity

Compute the required joint velocities near and at a singular configuration.
Observe how they blow up.

In [ ]:
# Desired EE velocity: 1 cm/s in x direction
xdot_d = np.array([0.01, 0.0, 0.0])

# Approach the fully-extended (boundary) singularity
q2_vals_sing = np.deg2rad(np.linspace(-75, 5, 50))
qdot_norms = []

for q2 in q2_vals_sing:
    q = np.array([0, q2, -q2*0.5, 0])   # approximate straight-arm path
    J = robot.jacob0(q)
    J_v = J[:3, :]
    # Use damped pseudo-inverse to avoid blow-up
    lam = 1e-4
    J_damp = J_v.T @ np.linalg.inv(J_v @ J_v.T + lam**2 * np.eye(3))
    qdot = J_damp @ xdot_d
    qdot_norms.append(np.linalg.norm(qdot))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(np.rad2deg(q2_vals_sing), qdot_norms, 'crimson', lw=2)
ax.set_xlabel('$q_2$ [deg]')
ax.set_ylabel('$\\|\\dot q\\|$ [rad/s]')
ax.set_title('Required joint speed for constant EE velocity\n(grows near singular configuration)')
plt.tight_layout()
plt.show()

---
## Part 6 — Tasks

**Task 1.** Compute FK for $q = (30°, -60°, 45°, -15°)$.
What is the end-effector position? What is the rotation matrix?
Extract the roll-pitch-yaw angles from the result.

**Task 2.** Find IK solutions for the target $T = (0.15, 0.10, 0.20)$ [m], no rotation.
How many distinct solutions exist?
For each solution, verify by computing FK and checking the position error.

**Task 3.** Using the Jacobian, compute the joint velocities needed to move
the end-effector at $\dot x = (0, 0.02, 0)$\,m/s (sideways)
at the configuration $q = (0°, -45°, 30°, 0°)$.
Are any joint velocities close to the Dynamixel limit (5.97\,rad/s)?

**Task 4.** Plot the manipulability map for $q_1$ vs $q_2$
(fix $q_3 = 30°$, $q_4 = 0°$).
Where is the most dexterous configuration?
What does it look like physically?